In [ ]:
import os
import csv
import numpy as np
import cv2
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision.models as models
import torchvision.transforms as transforms
from PIL import Image
from tqdm import tqdm
from collections import Counter

# ================= DEVICE =================
DEVICE = torch.device("mps" if torch.backends.mps.is_available()
                      else "cuda" if torch.cuda.is_available()
                      else "cpu")
print(f"Using device: {DEVICE}")

# ================= PATHS =================
MODEL_PATH = "/Users/dhruviramaiya/Downloads/Mtech Major Project/Multitask/best_optimized_multitask_model.pth"
MATURITY_TEST_DIR = "/Users/dhruviramaiya/Downloads/Mtech Major Project/Dataset/Maturity/train test val split for digital/test"
DISEASE_TEST_DIR  = "/Users/dhruviramaiya/Downloads/Mtech Major Project/Dataset/Guava_NewDataset/Guava_Preprocessed/test"
OUTPUT_DIR = "/Users/dhruviramaiya/Downloads/Mtech Major Project/XAI_Multitask/XAI_GradCAM/outputs"

os.makedirs(OUTPUT_DIR, exist_ok=True)

IMG_SIZE = 224

# ================= INSPECT CHECKPOINT =================
print("\n--- Inspecting checkpoint keys ---")
raw_state = torch.load(MODEL_PATH, map_location="cpu")

# Handle case where checkpoint is wrapped in a dict
if isinstance(raw_state, dict) and "state_dict" in raw_state:
    raw_state = raw_state["state_dict"]
elif isinstance(raw_state, dict) and "model" in raw_state:
    raw_state = raw_state["model"]

print("First 20 keys in checkpoint:")
for k in list(raw_state.keys())[:20]:
    print(f"  {k}  →  shape: {raw_state[k].shape}")

print(f"\nTotal keys in checkpoint: {len(raw_state)}")

# ================= MODEL =================
class CrossStitchUnit(nn.Module):
    def __init__(self):
        super().__init__()
        self.alpha = nn.Parameter(torch.tensor([[0.9,0.1],[0.1,0.9]], dtype=torch.float32))

    def forward(self,f1,f2):
        return (self.alpha[0,0]*f1 + self.alpha[0,1]*f2,
                self.alpha[1,0]*f1 + self.alpha[1,1]*f2)

class Model(nn.Module):
    def __init__(self):
        super().__init__()

        # Maturity branch
        self.m_conv1 = nn.Sequential(
            nn.Conv2d(3, 32, 3, padding=1),
            nn.ReLU(),
            nn.BatchNorm2d(32)
        )
        self.m_conv2 = nn.Sequential(
            nn.Conv2d(32, 64, 3, padding=1),
            nn.ReLU(),
            nn.BatchNorm2d(64)
        )

        # Disease branch
        self.d_conv1 = nn.Sequential(
            nn.Conv2d(3, 32, 3, padding=1),
            nn.ReLU(),
            nn.BatchNorm2d(32)
        )
        self.d_conv2 = nn.Sequential(
            nn.Conv2d(32, 64, 3, padding=1),
            nn.ReLU(),
            nn.BatchNorm2d(64)
        )

        # Cross-stitch
        self.cs1 = CrossStitchUnit()

        self.pool = nn.AdaptiveAvgPool2d((1,1))

        self.m_head = nn.Linear(64, 3)
        self.d_head = nn.Linear(64, 4)

    def forward(self, x):
        m = self.m_conv1(x)
        d = self.d_conv1(x)

        m, d = self.cs1(m, d)

        m = self.m_conv2(m)
        d = self.d_conv2(d)

        m = self.pool(m).view(x.size(0), -1)
        d = self.pool(d).view(x.size(0), -1)

        return self.m_head(m), self.d_head(d)

# ================= LOAD MODEL =================
model = Model()

print("\n--- Loading checkpoint STRICTLY ---")
missing, unexpected = model.load_state_dict(raw_state, strict=False)

print("Missing keys:", missing)
print("Unexpected keys:", unexpected)

# 🔴 CRITICAL CHECK
if len(missing) > 0 or len(unexpected) > 0:
    print("\n❌ ERROR: Model and checkpoint DO NOT match!")
    print("➡️ Fix architecture before proceeding.")
    exit()

print("✅ Model loaded successfully!")

model.to(DEVICE).eval()

# ================= DATASET AUTO DETECT =================
def get_classes(path):
    return sorted([d for d in os.listdir(path)
                   if os.path.isdir(os.path.join(path,d))])

# ================= CORE =================
def run_dataset(path, task, cam_obj):
    classes = get_classes(path)
    print(f"\n{task.upper()} classes detected: {classes}")

    results     = []
    pred_counter = []

    for i, cls in enumerate(classes):
        folder = os.path.join(path, cls)
        for img in tqdm(os.listdir(folder), desc=cls):
            try:
                p  = os.path.join(folder, img)
                im = Image.open(p).convert("RGB")
                x  = transform(im).unsqueeze(0).to(DEVICE)

                cam, pred = cam_obj.generate(x, task)
                pred_counter.append(pred)
                results.append({
                    "true":    cls,
                    "pred":    classes[pred] if pred < len(classes) else "UNK",
                    "correct": pred == i
                })
            except Exception as e:
                print(f"  Skipped {img}: {e}")
                continue

    print("\nPrediction distribution:", Counter(pred_counter))
    return results, classes

m_res, m_cls = run_dataset(MATURITY_TEST_DIR, "maturity", cam_m)
d_res, d_cls = run_dataset(DISEASE_TEST_DIR,  "disease",  cam_d)

# ================= ACCURACY =================
def report(res, classes, name):
    print(f"\n{'='*40}")
    print(f"{name.upper()} REPORT")
    print(f"{'='*40}")
    total   = len(res)
    correct = sum(r["correct"] for r in res)
    print(f"Overall accuracy: {round(100*correct/total, 2)}%")
    print()
    for c in classes:
        subset = [r for r in res if r["true"] == c]
        if not subset:
            continue
        acc = sum(r["correct"] for r in subset) / len(subset)
        # Confusion: what did it predict instead?
        preds = Counter(r["pred"] for r in subset)
        print(f"  {c}: {round(acc*100,2)}%  (n={len(subset)})  preds={dict(preds)}")

report(m_res, m_cls, "maturity")
report(d_res, d_cls, "disease")

: 